# Instamart Customer Intelligence & Support Copilot

**Capstone working notebook**

This notebook contains the exploratory work and execution steps supporting Parts A–D. The maintained Python scripts in `src/` are the reproducible implementation, while generated datasets, model artifacts, and evaluation outputs are stored in `data/` and `outputs/`.

**Run from the repository root** after installing `requirements.txt`. Some agent cells write checkpoint and ledger files; avoid rerunning recovery or refund-execution cells against an already populated state unless you intend to create a new run.


## Setup and Project Data Exploration

In [1]:
import sys

print("Python Version:")
print(sys.version)

print("\nPython Executable:")
print(sys.executable)

Python Version:
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]

Python Executable:
C:\Users\divya\anaconda3\python.exe


In [2]:
import pandas
import numpy
import sklearn
import sqlite3
import joblib

print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("SQLite:", sqlite3.sqlite_version)
print("Joblib:", joblib.__version__)

print("\nAll required libraries imported successfully!")

Pandas: 2.3.3
NumPy: 2.3.5
Scikit-learn: 1.7.2
SQLite: 3.51.0
Joblib: 1.5.2

All required libraries imported successfully!


In [3]:
import os

print("generate_data.py:", os.path.exists("generate_data.py"))
print("mock_llm.py:", os.path.exists("mock_llm.py"))

generate_data.py: True
mock_llm.py: True


In [4]:
%run generate_data.py

generated: customers=1200 products=40 orders_raw=13084 tickets=164


In [5]:
import os

for folder, subfolders, files in os.walk("data"):
    print("\nFolder:", folder)
    for file in files:
        print("  ", file)


Folder: data

Folder: data\policies
   delivery_policy.txt
   escalation_policy.txt
   payment_policy.txt
   privacy_policy.txt
   refund_policy.txt

Folder: data\raw
   customers.csv
   orders.csv
   policy_eval.csv
   products.csv
   tickets.csv


In [6]:
import pandas as pd
import os

customers = pd.read_csv("data/raw/customers.csv")
products = pd.read_csv("data/raw/products.csv")
orders = pd.read_csv("data/raw/orders.csv")
tickets = pd.read_csv("data/raw/tickets.csv")

print("CUSTOMERS:", customers.shape)
print("PRODUCTS:", products.shape)
print("ORDERS:", orders.shape)
print("TICKETS:", tickets.shape)

print("\nAll raw CSV files loaded successfully!")

CUSTOMERS: (1200, 5)
PRODUCTS: (40, 4)
ORDERS: (13084, 9)
TICKETS: (164, 5)

All raw CSV files loaded successfully!


In [7]:
datasets = {
    "Customers": customers,
    "Products": products,
    "Orders": orders,
    "Tickets": tickets
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name.upper(), "DATA AUDIT")
    print("=" * 60)

    print("\nDataset Shape:")
    print(df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nMissing Value Percentage:")
    print((df.isnull().sum() / len(df) * 100).round(2))


CUSTOMERS DATA AUDIT

Dataset Shape:
(1200, 5)

Data Types:
customer_id     object
city            object
signup_date     object
age            float64
gender          object
dtype: object

Missing Values:
customer_id     0
city            0
signup_date     0
age            61
gender         30
dtype: int64

Missing Value Percentage:
customer_id    0.00
city           0.00
signup_date    0.00
age            5.08
gender         2.50
dtype: float64

PRODUCTS DATA AUDIT

Dataset Shape:
(40, 4)

Data Types:
product_id      object
product_name    object
category        object
price_inr        int64
dtype: object

Missing Values:
product_id      0
product_name    0
category        0
price_inr       0
dtype: int64

Missing Value Percentage:
product_id      0.0
product_name    0.0
category        0.0
price_inr       0.0
dtype: float64

ORDERS DATA AUDIT

Dataset Shape:
(13084, 9)

Data Types:
order_id             object
customer_id          object
product_id           object
quantity         

In [8]:
print("========== CUSTOMERS ==========")

print("Blank ages:", customers["age"].isna().sum())
print("Ages outside 10-100:",
      ((customers["age"] < 10) | (customers["age"] > 100)).sum())

print("\nRaw city spellings:")
print(sorted(customers["city"].unique()))

print("\nRaw gender spellings:")
print(sorted(customers["gender"].dropna().unique()))


print("\n========== ORDERS ==========")

print("Exact duplicate rows:", orders.duplicated().sum())

print("Pending status rows:",
      (orders["status"].str.lower() == "pending").sum())

print("Distinct order IDs:", orders["order_id"].nunique())

print("Nonnumeric order_amount values:",
      pd.to_numeric(orders["order_amount"], errors="coerce").isna().sum()
      - orders["order_amount"].isna().sum())

print("Negative delivery minutes:",
      (orders["delivery_minutes"] < 0).sum())

print("Blank delivery minutes:",
      orders["delivery_minutes"].isna().sum())

========== CUSTOMERS ==========
Blank ages: 61
Ages outside 10-100: 6

Raw city spellings:
[' Bengaluru ', ' Chennai ', ' Delhi ', ' Hyderabad ', ' Mumbai ', ' Pune ', 'BENGALURU', 'Bengaluru', 'CHENNAI', 'Chennai', 'DELHI', 'Delhi', 'HYDERABAD', 'Hyderabad', 'MUMBAI', 'Mumbai', 'PUNE', 'Pune', 'bengaluru', 'chennai', 'delhi', 'hyderabad', 'mumbai', 'pune']

Raw gender spellings:
['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'f', 'female', 'm', 'male']

========== ORDERS ==========
Exact duplicate rows: 119
Pending status rows: 656
Distinct order IDs: 12309
Nonnumeric order_amount values: 67
Negative delivery minutes: 106
Blank delivery minutes: 2541


## Part A – Data Engineering, Cleaning and SQL Analysis

In [9]:
# Create a separate copy of the customer dataset
customers_clean = customers.copy()

# 1. Clean city names
customers_clean["city"] = (
    customers_clean["city"]
    .str.strip()
    .str.title()
)

# 2. Standardize gender values
gender_mapping = {
    "m": "male",
    "male": "male",
    "f": "female",
    "female": "female"
}

customers_clean["gender"] = (
    customers_clean["gender"]
    .str.strip()
    .str.lower()
    .map(gender_mapping)
    .fillna("unknown")
)

# 3. Replace invalid ages with missing values
customers_clean.loc[
    (customers_clean["age"] < 10) |
    (customers_clean["age"] > 100),
    "age"
] = float("nan")

# 4. Fill missing ages using city-wise median
customers_clean["age"] = (
    customers_clean["age"]
    .fillna(
        customers_clean.groupby("city")["age"]
        .transform("median")
    )
)

# 5. Convert signup_date into datetime
customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"]
)

print("Customer data cleaning completed!")
print("\nFirst five cleaned records:")
print(customers_clean.head())

Customer data cleaning completed!

First five cleaned records:
  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    male
1       C0002      Delhi  2024-02-17  31.0    male
2       C0003  Bengaluru  2025-02-18  18.0  female
3       C0004  Bengaluru  2024-06-21  24.0  female
4       C0005  Bengaluru  2024-10-01  25.0  female


In [10]:
print("========== CUSTOMER CLEANING VALIDATION ==========")

print("\n1. City counts:")
print(customers_clean["city"].value_counts())

print("\n2. Gender counts:")
print(customers_clean["gender"].value_counts())

print("\n3. Missing ages:")
print(customers_clean["age"].isna().sum())

print("\n4. Minimum age:")
print(customers_clean["age"].min())

print("\n5. Maximum age:")
print(customers_clean["age"].max())

print("\n6. Signup date data type:")
print(customers_clean["signup_date"].dtype)

print("\n7. Unique cities:")
print(customers_clean["city"].nunique())

========== CUSTOMER CLEANING VALIDATION ==========

1. City counts:
city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184
Name: count, dtype: int64

2. Gender counts:
gender
female     610
male       560
unknown     30
Name: count, dtype: int64

3. Missing ages:
0

4. Minimum age:
18.0

5. Maximum age:
60.0

6. Signup date data type:
datetime64[ns]

7. Unique cities:
6


In [11]:
import os

os.makedirs("data/clean", exist_ok=True)

customers_clean.to_csv(
    "data/clean/customers.csv",
    index=False
)

print("Cleaned customer data saved successfully!")
print("File: data/clean/customers.csv")

Cleaned customer data saved successfully!
File: data/clean/customers.csv


In [12]:
# Create a separate copy of raw orders
orders_clean = orders.copy()

print("Original orders:", orders.shape)
print("Working copy:", orders_clean.shape)

print("\nOrder columns:")
print(orders_clean.columns.tolist())

print("\nFirst five records:")
print(orders_clean.head())

Original orders: (13084, 9)
Working copy: (13084, 9)

Order columns:
['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes']

First five records:
     order_id customer_id product_id  quantity             order_ts  \
0  ORD-108250       C0792       P004         2  2025-09-01 12:24:59   
1  ORD-101939       C0190       P003         1  2025-08-17 07:07:23   
2  ORD-108765       C0841       P033         1  2025-07-13 09:42:32   
3  ORD-108861       C0852       P010         1  2025-07-04 11:23:24   
4  ORD-109150       C0877       P018         2  2025-05-07 15:35:01   

      status  discount_pct order_amount  delivery_minutes  
0  completed             0        310.0              17.0  
1  completed             5       232.75              21.0  
2  completed            10       1349.1              27.0  
3  completed             0         45.0              30.0  
4  completed            20         96.0              28

In [13]:
# Convert order_ts to datetime
orders_clean["order_ts"] = pd.to_datetime(
    orders_clean["order_ts"]
)

print("Order timestamp converted successfully!")

print("\nData type:")
print(orders_clean["order_ts"].dtype)

print("\nFirst five timestamps:")
print(orders_clean["order_ts"].head())

Order timestamp converted successfully!

Data type:
datetime64[ns]

First five timestamps:
0   2025-09-01 12:24:59
1   2025-08-17 07:07:23
2   2025-07-13 09:42:32
3   2025-07-04 11:23:24
4   2025-05-07 15:35:01
Name: order_ts, dtype: datetime64[ns]


In [14]:
# Check duplicate count before removal
print("Duplicates before cleaning:",
      orders_clean.duplicated().sum())

# Remove exact duplicate rows
orders_clean = orders_clean.drop_duplicates()

# Reset row index
orders_clean = orders_clean.reset_index(drop=True)

print("\nDuplicates after cleaning:",
      orders_clean.duplicated().sum())

print("Rows remaining:", len(orders_clean))

Duplicates before cleaning: 119

Duplicates after cleaning: 0
Rows remaining: 12965


In [15]:
# Sort orders by timestamp
orders_clean = orders_clean.sort_values(
    by="order_ts"
)

# Keep the latest record for each order_id
orders_clean = orders_clean.drop_duplicates(
    subset="order_id",
    keep="last"
)

# Reset index
orders_clean = orders_clean.reset_index(drop=True)

print("Repeated order IDs handled successfully!")

print("\nTotal rows:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Duplicate order IDs:", orders_clean["order_id"].duplicated().sum())

Repeated order IDs handled successfully!

Total rows: 12309
Unique order IDs: 12309
Duplicate order IDs: 0


In [16]:
# Convert order_amount to numeric
orders_clean["order_amount"] = pd.to_numeric(
    orders_clean["order_amount"],
    errors="coerce"
)

print("Order amount converted successfully!")

print("\nData type:")
print(orders_clean["order_amount"].dtype)

print("\nMissing order amounts:")
print(orders_clean["order_amount"].isna().sum())

print("\nNegative order amounts:")
print((orders_clean["order_amount"] < 0).sum())

Order amount converted successfully!

Data type:
float64

Missing order amounts:
112

Negative order amounts:
0


In [17]:
# Inspect rows with missing order amounts
missing_amounts = orders_clean[
    orders_clean["order_amount"].isna()
]

print("Missing amount records:", len(missing_amounts))

print("\nMissing amount records:")
print(missing_amounts.head(10))

print("\nStatus distribution:")
print(missing_amounts["status"].value_counts())

Missing amount records: 112

Missing amount records:
        order_id customer_id product_id  quantity            order_ts  \
89    ORD-101209       C0119       P007         3 2025-04-02 09:28:14   
143   ORD-104216       C0396       P004         1 2025-04-02 20:15:35   
307   ORD-103504       C0331       P038         1 2025-04-05 18:36:19   
442   ORD-100697       C0072       P001         1 2025-04-07 16:51:08   
619   ORD-103779       C0352       P013         2 2025-04-10 09:24:12   
676   ORD-107941       C0758       P018         3 2025-04-10 21:45:52   
789   ORD-109149       C0877       P003         1 2025-04-12 18:35:33   
814   ORD-107374       C0708       P017         1 2025-04-13 08:02:15   
859   ORD-108913       C0855       P019         1 2025-04-13 17:37:39   
1032  ORD-110243       C0986       P022         1 2025-04-16 08:20:01   

         status  discount_pct  order_amount  delivery_minutes  
89    completed             0           NaN              23.0  
143   completed

In [18]:
print("========== DELIVERY TIME VALIDATION ==========")

print("Missing delivery times:",
      orders_clean["delivery_minutes"].isna().sum())

print("Negative delivery times:",
      (orders_clean["delivery_minutes"] < 0).sum())

print("\nDelivery time statistics:")
print(orders_clean["delivery_minutes"].describe())

print("\nStatus counts:")
print(orders_clean["status"].value_counts())

========== DELIVERY TIME VALIDATION ==========
Missing delivery times: 1868
Negative delivery times: 105

Delivery time statistics:
count    10441.000000
mean        27.169716
std         13.662402
min        -10.000000
25%         19.000000
50%         25.000000
75%         31.000000
max         75.000000
Name: delivery_minutes, dtype: float64

Status counts:
status
completed    11110
cancelled     1199
Name: count, dtype: int64


In [19]:
# Inspect negative delivery times
negative_delivery = orders_clean[
    orders_clean["delivery_minutes"] < 0
]

print("Negative delivery records:", len(negative_delivery))

print("\nStatus distribution:")
print(negative_delivery["status"].value_counts())

print("\nSample negative delivery records:")
print(negative_delivery.head(10))

Negative delivery records: 105

Status distribution:
status
completed    105
Name: count, dtype: int64

Sample negative delivery records:
        order_id customer_id product_id  quantity            order_ts  \
237   ORD-102608       C0246       P005         3 2025-04-04 14:13:44   
555   ORD-106508       C0622       P028         1 2025-04-09 08:09:00   
708   ORD-106708       C0640       P011         3 2025-04-11 12:59:30   
797   ORD-101308       C0127       P030         1 2025-04-12 19:40:33   
1045  ORD-100608       C0066       P016         3 2025-04-16 11:48:30   
1072  ORD-108108       C0775       P038         1 2025-04-16 19:59:03   
1097  ORD-109008       C0866       P012         1 2025-04-17 08:50:15   
1163  ORD-108808       C0845       P008         3 2025-04-17 22:29:34   
1300  ORD-108608       C0819       P037         1 2025-04-20 07:56:40   
1518  ORD-110908       C1064       P022         1 2025-04-23 13:32:29   

         status  discount_pct  order_amount  delivery_minu

In [20]:
# Get product prices using product_id
price_map = products.set_index("product_id")["price_inr"]

# Identify missing order amounts
missing_mask = orders_clean["order_amount"].isna()

# Calculate missing amounts using the assignment formula
orders_clean.loc[missing_mask, "order_amount"] = (
    orders_clean.loc[missing_mask, "product_id"].map(price_map)
    * orders_clean.loc[missing_mask, "quantity"]
    * (1 - orders_clean.loc[missing_mask, "discount_pct"] / 100)
).round(2)

print("Missing order amounts filled successfully!")

print("\nRemaining missing amounts:",
      orders_clean["order_amount"].isna().sum())

print("\nOrder amount data type:",
      orders_clean["order_amount"].dtype)

print("\nCompleted orders revenue:")
print(
    round(
        orders_clean.loc[
            orders_clean["status"] == "completed",
            "order_amount"
        ].sum(),
        2
    )
)

Missing order amounts filled successfully!

Remaining missing amounts: 0

Order amount data type: float64

Completed orders revenue:
2998668.8


In [21]:
# Step 1: Replace negative delivery times with NaN
orders_clean.loc[
    orders_clean["delivery_minutes"] < 0,
    "delivery_minutes"
] = float("nan")

# Step 2: Get product categories
orders_clean["category"] = orders_clean["product_id"].map(
    products.set_index("product_id")["category"]
)

# Step 3: Calculate category-wise median using completed orders
completed_mask = orders_clean["status"] == "completed"

category_medians = (
    orders_clean.loc[completed_mask]
    .groupby("category")["delivery_minutes"]
    .transform("median")
)

# Step 4: Fill missing delivery times only for completed orders
orders_clean.loc[completed_mask, "delivery_minutes"] = (
    orders_clean.loc[completed_mask, "delivery_minutes"]
    .fillna(category_medians)
)

print("Delivery time cleaning completed!")

print("\nMissing delivery times among completed orders:",
      orders_clean.loc[completed_mask, "delivery_minutes"].isna().sum())

print("\nMissing delivery times among cancelled orders:",
      orders_clean.loc[~completed_mask, "delivery_minutes"].isna().sum())

print("\nNegative delivery times:",
      (orders_clean["delivery_minutes"] < 0).sum())

Delivery time cleaning completed!

Missing delivery times among completed orders: 0

Missing delivery times among cancelled orders: 1199

Negative delivery times: 0


In [22]:
completed_mask = orders_clean["status"] == "completed"
cancelled_mask = orders_clean["status"] == "cancelled"

print("Orders shape:", orders_clean.shape)

print("\nOrder status counts:")
print(orders_clean["status"].value_counts())

print("\nPending orders:",
      (orders_clean["status"] == "pending").sum())

print("\nMissing order amounts:",
      orders_clean["order_amount"].isna().sum())

print("\nMissing delivery in completed orders:",
      orders_clean.loc[completed_mask, "delivery_minutes"].isna().sum())

print("\nMissing delivery in cancelled orders:",
      orders_clean.loc[cancelled_mask, "delivery_minutes"].isna().sum())

print("\nCompleted orders revenue:",
      round(orders_clean.loc[completed_mask, "order_amount"].sum(), 2))

Orders shape: (12309, 10)

Order status counts:
status
completed    11110
cancelled     1199
Name: count, dtype: int64

Pending orders: 0

Missing order amounts: 0

Missing delivery in completed orders: 0

Missing delivery in cancelled orders: 1199

Completed orders revenue: 2998668.8


In [23]:
orders_clean = orders_clean.drop(columns=["category"])

print("Final orders shape:", orders_clean.shape)
print("\nFinal columns:")
print(orders_clean.columns.tolist())

Final orders shape: (12309, 9)

Final columns:
['order_id', 'customer_id', 'product_id', 'quantity', 'order_ts', 'status', 'discount_pct', 'order_amount', 'delivery_minutes']


In [24]:
# Create a clean copy of products
products_clean = products.copy()

# Save cleaned products
products_clean.to_csv("data/clean/products.csv", index=False)

# Save cleaned orders
orders_clean.to_csv("data/clean/orders.csv", index=False)

print("Cleaned datasets saved successfully!")
print("1. data/clean/customers.csv")
print("2. data/clean/products.csv")
print("3. data/clean/orders.csv")

Cleaned datasets saved successfully!
1. data/clean/customers.csv
2. data/clean/products.csv
3. data/clean/orders.csv


In [25]:
import sqlite3

# Create database connection
conn = sqlite3.connect("data/instamart.db")

# Prepare orders for SQLite
orders_db = orders_clean.copy()

# Store order timestamp in the required text format
orders_db["order_ts"] = orders_db["order_ts"].dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)

# Create three database tables
customers_clean.to_sql(
    "customers", conn, if_exists="replace", index=False
)

products_clean.to_sql(
    "products", conn, if_exists="replace", index=False
)

orders_db.to_sql(
    "orders", conn, if_exists="replace", index=False
)

conn.close()

print("SQLite database created successfully!")
print("Database: data/instamart.db")
print("Tables: customers, products, orders")

SQLite database created successfully!
Database: data/instamart.db
Tables: customers, products, orders


In [26]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/instamart.db")

# Check available tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print("Database tables:")
print(tables)

# Check record counts
for table in ["customers", "products", "orders"]:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS total_records FROM {table}",
        conn
    )
    print(f"\n{table}: {result.iloc[0, 0]} records")

conn.close()

Database tables:
        name
0  customers
1   products
2     orders

customers: 1200 records

products: 40 records

orders: 12309 records


In [27]:
import os

os.makedirs("src", exist_ok=True)

print("src folder created successfully!")

src folder created successfully!


In [28]:
validation_code = '''
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/instamart.db")

customers = pd.read_sql_query("SELECT * FROM customers", conn)
products = pd.read_sql_query("SELECT * FROM products", conn)
orders = pd.read_sql_query("SELECT * FROM orders", conn)

conn.close()

checks = []

checks.append(len(customers) == 1200)
checks.append(len(products) == 40)
checks.append(len(orders) == 12309)
checks.append(customers["customer_id"].is_unique)
checks.append(products["product_id"].is_unique)
checks.append(orders["order_id"].is_unique)
checks.append((orders["status"] == "pending").sum() == 0)
checks.append(orders["order_amount"].isna().sum() == 0)

completed = orders[orders["status"] == "completed"]

checks.append(completed["delivery_minutes"].isna().sum() == 0)
checks.append(round(completed["order_amount"].sum(), 2) == 2998668.80)

if all(checks):
    print("ALL CHECKPOINTS PASSED")
else:
    print("VALIDATION FAILED")
    for i, result in enumerate(checks, start=1):
        print(f"Checkpoint {i}: {result}")
'''

with open("src/validate.py", "w") as file:
    file.write(validation_code)

print("validate.py created successfully!")

validate.py created successfully!


In [29]:
%run src/validate.py

ALL CHECKPOINTS PASSED


In [30]:
import os

os.makedirs("sql", exist_ok=True)

print("sql folder created successfully!")

sql folder created successfully!


In [31]:
sql_code = """
-- Q1: WHERE and HAVING
SELECT
    customer_id,
    SUM(order_amount) AS total_spend
FROM orders
WHERE status = 'completed'
  AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000;


-- Q2: Three-table JOIN - Revenue by city and category
SELECT
    c.city,
    p.category,
    ROUND(SUM(o.order_amount), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.city, p.category
ORDER BY c.city, p.category;


-- Q3: Scalar subquery - Products above average price
SELECT
    product_id,
    product_name,
    price_inr
FROM products
WHERE price_inr > (
    SELECT AVG(price_inr)
    FROM products
)
ORDER BY price_inr DESC;


-- Q4: LEFT JOIN - Customers without completed orders
SELECT
    c.customer_id,
    c.city
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'completed'
WHERE o.order_id IS NULL;


-- Q5: CTE - Customers spending above average
WITH customer_spend AS (
    SELECT
        customer_id,
        SUM(order_amount) AS total_spend
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
),
average_spend AS (
    SELECT AVG(total_spend) AS avg_spend
    FROM customer_spend
)
SELECT
    cs.customer_id,
    ROUND(cs.total_spend, 2) AS total_spend,
    ROUND(a.avg_spend, 2) AS average_spend
FROM customer_spend cs
CROSS JOIN average_spend a
WHERE cs.total_spend > a.avg_spend
ORDER BY cs.total_spend DESC;


-- Q6: Window functions - Customer spending rankings by city
WITH customer_spend AS (
    SELECT
        c.customer_id,
        c.city,
        SUM(o.order_amount) AS total_spend
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.city
)
SELECT
    customer_id,
    city,
    ROUND(total_spend, 2) AS total_spend,
    RANK() OVER (
        PARTITION BY city ORDER BY total_spend DESC
    ) AS spend_rank,
    DENSE_RANK() OVER (
        PARTITION BY city ORDER BY total_spend DESC
    ) AS dense_rank,
    ROW_NUMBER() OVER (
        PARTITION BY city ORDER BY total_spend DESC
    ) AS row_number
FROM customer_spend
ORDER BY city, spend_rank;


-- Q7: Running total - Daily completed revenue
WITH daily_revenue AS (
    SELECT
        SUBSTR(order_ts, 1, 10) AS day,
        SUM(order_amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY SUBSTR(order_ts, 1, 10)
)
SELECT
    day,
    ROUND(revenue, 2) AS daily_revenue,
    ROUND(
        SUM(revenue) OVER (ORDER BY day),
        2
    ) AS cumulative_revenue
FROM daily_revenue
ORDER BY day;
"""

with open("sql/queries.sql", "w", encoding="utf-8") as file:
    file.write(sql_code)

print("queries.sql created successfully!")
print("Total SQL queries: 7")

queries.sql created successfully!
Total SQL queries: 7


In [32]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/instamart.db")

q1 = pd.read_sql_query("""
SELECT
    customer_id,
    SUM(order_amount) AS total_spend
FROM orders
WHERE status = 'completed'
  AND order_amount > 500
GROUP BY customer_id
HAVING SUM(order_amount) > 8000;
""", conn)

print("Q1 Results:")
print(q1)

print("\nTotal customers:", len(q1))

conn.close()

Q1 Results:
   customer_id  total_spend
0        C0082     12317.70
1        C0130      9170.40
2        C0147      9141.50
3        C0167      8718.55
4        C0267     12150.25
5        C0291     12150.70
6        C0308      9872.40
7        C0332     10779.50
8        C0407      9451.75
9        C0424      8294.20
10       C0426      9884.60
11       C0454      8299.15
12       C0488     11825.50
13       C0512     11617.65
14       C0525     12241.30
15       C0543     10636.10
16       C0693     13776.55
17       C0696      8941.75
18       C0699     11693.90
19       C0743     12266.85
20       C0775     10263.55
21       C0819      8695.30
22       C0916      8395.35
23       C0930      9194.00
24       C0942      8643.60
25       C1000      8462.20
26       C1024     10160.10
27       C1136      8983.75
28       C1146      8993.05
29       C1147      9740.80

Total customers: 30


In [33]:
conn = sqlite3.connect("data/instamart.db")

q2 = pd.read_sql_query("""
SELECT
    c.city,
    p.category,
    ROUND(SUM(o.order_amount), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.city, p.category
ORDER BY c.city, p.category;
""", conn)

print("Q2 Results:")
print(q2)

print("\nTotal rows:", len(q2))
print("Grand total revenue:", round(q2["total_revenue"].sum(), 2))

print("\nHighest revenue city/category:")
print(q2.loc[q2["total_revenue"].idxmax()])

conn.close()

Q2 Results:
         city       category  total_revenue
0   Bengaluru          Dairy       87075.05
1   Bengaluru    Electronics      233909.45
2   Bengaluru        Grocery      110915.65
3   Bengaluru  Personal Care       20193.85
4   Bengaluru         Snacks       71549.40
5     Chennai          Dairy       78301.80
6     Chennai    Electronics      195508.80
7     Chennai        Grocery       93817.45
8     Chennai  Personal Care       27675.40
9     Chennai         Snacks       68325.70
10      Delhi          Dairy      117271.70
11      Delhi    Electronics      158549.10
12      Delhi        Grocery      112658.05
13      Delhi  Personal Care       23471.50
14      Delhi         Snacks       90879.45
15  Hyderabad          Dairy       83731.00
16  Hyderabad    Electronics      228913.10
17  Hyderabad        Grocery      109475.40
18  Hyderabad  Personal Care       24800.05
19  Hyderabad         Snacks       81670.35
20     Mumbai          Dairy       75209.85
21     Mumbai    Ele

In [34]:
conn = sqlite3.connect("data/instamart.db")

q3 = pd.read_sql_query("""
SELECT
    product_id,
    product_name,
    price_inr
FROM products
WHERE price_inr > (
    SELECT AVG(price_inr)
    FROM products
)
ORDER BY price_inr DESC;
""", conn)

print("Q3 Results:")
print(q3)

print("\nTotal products above average price:", len(q3))

conn.close()

Q3 Results:
   product_id         product_name  price_inr
0        P038    Bluetooth Speaker       1999
1        P033     Wireless Earbuds       1499
2        P034  Power Bank 10000mAh       1299
3        P036    Phone Charger 20W        799
4        P037           Smart Bulb        599
5        P040     Memory Card 64GB        549
6        P032        Sunscreen 50g        399
7        P030    Body Lotion 400ml        350
8        P039          Phone Stand        349
9        P014           Ghee 500ml        340
10       P035       USB-C Cable 1m        299

Total products above average price: 11


In [35]:
conn = sqlite3.connect("data/instamart.db")

q4 = pd.read_sql_query("""
SELECT
    c.customer_id,
    c.city
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
    AND o.status = 'completed'
WHERE o.order_id IS NULL;
""", conn)

print("Q4 Results:")
print(q4)

print("\nCustomers without completed orders:", len(q4))

conn.close()

Q4 Results:
   customer_id       city
0        C0029  Hyderabad
1        C0037       Pune
2        C0093       Pune
3        C0115  Bengaluru
4        C0120  Bengaluru
5        C0151       Pune
6        C0354    Chennai
7        C0363    Chennai
8        C0412      Delhi
9        C0461  Bengaluru
10       C0477     Mumbai
11       C0500  Hyderabad
12       C0607       Pune
13       C0625      Delhi
14       C0654  Hyderabad
15       C0724    Chennai
16       C0759      Delhi
17       C0827    Chennai
18       C0835  Hyderabad
19       C1041      Delhi
20       C1052      Delhi
21       C1083       Pune
22       C1111       Pune
23       C1165  Bengaluru
24       C1189  Bengaluru

Customers without completed orders: 25


In [36]:
conn = sqlite3.connect("data/instamart.db")

q5 = pd.read_sql_query("""
WITH customer_spend AS (
    SELECT
        customer_id,
        SUM(order_amount) AS total_spend
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
),
average_spend AS (
    SELECT AVG(total_spend) AS avg_spend
    FROM customer_spend
)
SELECT
    cs.customer_id,
    ROUND(cs.total_spend, 2) AS total_spend,
    ROUND(a.avg_spend, 2) AS average_spend
FROM customer_spend cs
CROSS JOIN average_spend a
WHERE cs.total_spend > a.avg_spend
ORDER BY cs.total_spend DESC;
""", conn)

print("Q5 Results:")
print(q5.head(10))

print("\nTotal customers above average:", len(q5))
print("Average spending:", q5["average_spend"].iloc[0])

conn.close()

Q5 Results:
  customer_id  total_spend  average_spend
0       C0693     14766.55        2552.06
1       C0291     13742.60        2552.06
2       C0488     13187.50        2552.06
3       C0082     12856.20        2552.06
4       C0743     12730.05        2552.06
5       C0525     12719.70        2552.06
6       C0699     12471.80        2552.06
7       C0267     12210.25        2552.06
8       C0332     11806.40        2552.06
9       C0512     11617.65        2552.06

Total customers above average: 550
Average spending: 2552.06


In [37]:
conn = sqlite3.connect("data/instamart.db")

q6 = pd.read_sql_query("""
WITH customer_spend AS (
    SELECT
        c.customer_id,
        c.city,
        SUM(o.order_amount) AS total_spend
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.customer_id, c.city
)
SELECT
    customer_id,
    city,
    ROUND(total_spend, 2) AS total_spend,

    RANK() OVER(
        PARTITION BY city
        ORDER BY total_spend DESC
    ) AS spend_rank,

    DENSE_RANK() OVER(
        PARTITION BY city
        ORDER BY total_spend DESC
    ) AS dense_rank,

    ROW_NUMBER() OVER(
        PARTITION BY city
        ORDER BY total_spend DESC
    ) AS row_number

FROM customer_spend
ORDER BY city, spend_rank;
""", conn)

print("Q6 Results:")
print(q6.head(15))

print("\nTotal customers ranked:", len(q6))

print("\nTop customer in each city:")
print(q6.groupby("city").first())

print("\nRANK and ROW_NUMBER differences:",
      (q6["spend_rank"] != q6["row_number"]).sum())

conn.close()

Q6 Results:
   customer_id       city  total_spend  spend_rank  dense_rank  row_number
0        C0082  Bengaluru     12856.20           1           1           1
1        C0525  Bengaluru     12719.70           2           2           2
2        C0699  Bengaluru     12471.80           3           3           3
3        C0543  Bengaluru     11195.10           4           4           4
4        C1147  Bengaluru     10156.90           5           5           5
5        C0407  Bengaluru      9945.85           6           6           6
6        C0942  Bengaluru      8643.60           7           7           7
7        C1126  Bengaluru      8595.70           8           8           8
8        C0916  Bengaluru      8444.85           9           9           9
9        C0098  Bengaluru      8151.20          10          10          10
10       C1102  Bengaluru      7503.60          11          11          11
11       C0230  Bengaluru      7286.95          12          12          12
12       C117

In [38]:
conn = sqlite3.connect("data/instamart.db")

q7 = pd.read_sql_query("""
WITH daily_revenue AS (
    SELECT
        SUBSTR(order_ts, 1, 10) AS day,
        SUM(order_amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY SUBSTR(order_ts, 1, 10)
)
SELECT
    day,
    ROUND(revenue, 2) AS daily_revenue,
    ROUND(
        SUM(revenue) OVER(ORDER BY day),
        2
    ) AS cumulative_revenue
FROM daily_revenue
ORDER BY day;
""", conn)

print("Q7 Results:")
print(q7.head(10))

print("\nTotal daily records:", len(q7))

print("\nLast record:")
print(q7.tail(1))

conn.close()

Q7 Results:
          day  daily_revenue  cumulative_revenue
0  2025-04-01       16876.50            16876.50
1  2025-04-02       19854.55            36731.05
2  2025-04-03       28048.35            64779.40
3  2025-04-04       14249.60            79029.00
4  2025-04-05       16031.50            95060.50
5  2025-04-06       26323.80           121384.30
6  2025-04-07       15606.90           136991.20
7  2025-04-08       17055.45           154046.65
8  2025-04-09       20403.35           174450.00
9  2025-04-10       15359.90           189809.90

Total daily records: 183

Last record:
            day  daily_revenue  cumulative_revenue
182  2025-09-30        15503.1           2998668.8


In [39]:
# Merge completed orders with customers and products
completed_orders = orders_clean[
    orders_clean["status"] == "completed"
].copy()

merged_data = (
    completed_orders
    .merge(customers_clean[["customer_id", "city"]], on="customer_id")
    .merge(products[["product_id", "category"]], on="product_id")
)

# Create pivot table
pivot_result = pd.pivot_table(
    merged_data,
    values="order_amount",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True
)

print("Pandas Pivot Table:")
print(pivot_result)

print("\nPandas Grand Total:",
      round(pivot_result.loc["All", "All"], 2))

print("SQL Grand Total:",
      round(q2["total_revenue"].sum(), 2))

Pandas Pivot Table:
category       Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                     
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   
All        531938.95   1219248.10  614293.30      168433.95  464754.50   

category          All  
city                   
Bengaluru   523643.40  
Chennai     463629.15  
Delhi       502829.80  
Hyderabad   528589.90  
Mumbai      479058.50  
Pune        500918.05  
All        2998668.80  

Pandas Grand Total: 2998668.8
SQL Grand Total: 2998668.8


In [40]:
city_analysis = (
    completed_orders
    .merge(
        customers_clean[["customer_id", "city"]],
        on="customer_id"
    )
    .groupby("city")
    .agg(
        total_revenue=("order_amount", "sum"),
        average_order_amount=("order_amount", "mean"),
        order_count=("order_amount", "count")
    )
    .round(2)
    .reset_index()
)

print("City-wise Revenue Analysis:")
print(city_analysis)

City-wise Revenue Analysis:
        city  total_revenue  average_order_amount  order_count
0  Bengaluru      523643.40                276.47         1894
1    Chennai      463629.15                272.72         1700
2      Delhi      502829.80                243.62         2064
3  Hyderabad      528589.90                284.95         1855
4     Mumbai      479058.50                297.00         1613
5       Pune      500918.05                252.48         1984


In [41]:
amounts = completed_orders["order_amount"]

# Basic statistics
mean_value = amounts.mean()
median_value = amounts.median()
skew_value = amounts.skew()

# Quartiles
q1 = amounts.quantile(0.25)
q2 = amounts.quantile(0.50)
q3 = amounts.quantile(0.75)

# IQR
iqr = q3 - q1

# Outlier fences
lower_fence = q1 - (1.5 * iqr)
upper_fence = q3 + (1.5 * iqr)

# Outlier counts
lower_outliers = (amounts < lower_fence).sum()
upper_outliers = (amounts > upper_fence).sum()

print("STATISTICAL ANALYSIS")
print("--------------------")
print("Mean:", round(mean_value, 2))
print("Median:", round(median_value, 2))
print("Skewness:", round(skew_value, 2))

print("\nQuartiles")
print("Q1:", round(q1, 2))
print("Q2:", round(q2, 2))
print("Q3:", round(q3, 2))

print("\nIQR:", round(iqr, 2))

print("\nOutlier Analysis")
print("Lower fence:", round(lower_fence, 2))
print("Upper fence:", round(upper_fence, 2))
print("Lower outliers:", lower_outliers)
print("Upper outliers:", upper_outliers)
print("Total outliers:", lower_outliers + upper_outliers)

STATISTICAL ANALYSIS
--------------------
Mean: 269.91
Median: 135.0
Skewness: 4.61

Quartiles
Q1: 64.8
Q2: 135.0
Q3: 297.0

IQR: 232.2

Outlier Analysis
Lower fence: -283.5
Upper fence: 645.3
Lower outliers: 0
Upper outliers: 897
Total outliers: 897


In [42]:
correlation_matrix = completed_orders[
    [
        "order_amount",
        "discount_pct",
        "quantity",
        "delivery_minutes"
    ]
].corr()

print("CORRELATION MATRIX")
print("------------------")
print(correlation_matrix.round(3))

# Find the strongest relationship, excluding self-correlations
corr_pairs = (
    correlation_matrix
    .where(
        ~__import__("numpy").eye(
            len(correlation_matrix),
            dtype=bool
        )
    )
    .stack()
)

strongest_pair = corr_pairs.abs().idxmax()

print("\nStrongest correlation pair:", strongest_pair)
print("Correlation value:", round(
    correlation_matrix.loc[strongest_pair[0], strongest_pair[1]], 3
))

CORRELATION MATRIX
------------------
                  order_amount  discount_pct  quantity  delivery_minutes
order_amount             1.000        -0.095     0.087            -0.003
discount_pct            -0.095         1.000    -0.000            -0.010
quantity                 0.087        -0.000     1.000            -0.013
delivery_minutes        -0.003        -0.010    -0.013             1.000

Strongest correlation pair: ('order_amount', 'discount_pct')
Correlation value: -0.095


### Part A – Reproducible Script Execution and EDA

In [43]:
%run src/clean_data.py

INSTAMART DATA CLEANING STARTED

Raw datasets loaded successfully
Customers: (1200, 5)
Products: (40, 4)
Orders: (13084, 9)

Customer cleaning completed
Product data saved
Order cleaning completed

SQLite database created successfully

FINAL CLEANING SUMMARY
----------------------
Customers: 1200
Products: 40
Orders: 12309
Completed orders: 11110
Cancelled orders: 1199
Missing completed-order amounts: 0

CLEANING SCRIPT COMPLETED SUCCESSFULLY!


In [44]:
%run src/validate.py

ALL CHECKPOINTS PASSED
ALL CHECKPOINTS PASSED


In [45]:
%run src/run_sql.py

TOTAL SQL QUERIES: 7

QUERY 1 RESULTS
  customer_id  total_spend
0       C0082     12317.70
1       C0130      9170.40
2       C0147      9141.50
3       C0167      8718.55
4       C0267     12150.25
5       C0291     12150.70
6       C0308      9872.40
7       C0332     10779.50
8       C0407      9451.75
9       C0424      8294.20

Total rows: 30

QUERY 2 RESULTS
        city       category  total_revenue
0  Bengaluru          Dairy       87075.05
1  Bengaluru    Electronics      233909.45
2  Bengaluru        Grocery      110915.65
3  Bengaluru  Personal Care       20193.85
4  Bengaluru         Snacks       71549.40
5    Chennai          Dairy       78301.80
6    Chennai    Electronics      195508.80
7    Chennai        Grocery       93817.45
8    Chennai  Personal Care       27675.40
9    Chennai         Snacks       68325.70

Total rows: 30

QUERY 3 RESULTS
  product_id         product_name  price_inr
0       P038    Bluetooth Speaker       1999
1       P033     Wireless Earbuds   

In [46]:
%run src/eda_analysis.py

INSTAMART EXPLORATORY DATA ANALYSIS

PANDAS PIVOT TABLE
category       Dairy  Electronics    Grocery  Personal Care     Snacks  \
city                                                                     
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   
All        531938.95   1219248.10  614293.30      168433.95  464754.50   

category          All  
city                   
Bengaluru   523643.40  
Chennai     463629.15  
Delhi       502829.80  
Hyderabad   528589.90  
Mumbai      479058.50  
Pune        500918.05  
All        2998668.80  

Grand Total: 2998668.8

CITY-WISE ANALYSIS
           total_

## Part B – Customer Intelligence and Lapse Prediction

In [47]:
%run src/customer_segmentation.py

INSTAMART PART B - CUSTOMER SEGMENTATION

CUSTOMER DATA
Shape: (1200, 5)
  customer_id       city signup_date   age  gender
0       C0001    Chennai  2024-01-13  35.0    male
1       C0002      Delhi  2024-02-17  31.0    male
2       C0003  Bengaluru  2025-02-18  18.0  female
3       C0004  Bengaluru  2024-06-21  24.0  female
4       C0005  Bengaluru  2024-10-01  25.0  female

ORDER DATA
Shape: (12309, 9)
     order_id customer_id product_id  quantity            order_ts     status  \
0  ORD-110093       C0967       P020         1 2025-04-01 07:08:28  completed   
1  ORD-111396       C1117       P036         2 2025-04-01 07:11:59  completed   
2  ORD-102006       C0194       P009         1 2025-04-01 07:12:43  cancelled   
3  ORD-105921       C0569       P005         2 2025-04-01 07:54:43  completed   
4  ORD-108115       C0776       P012         2 2025-04-01 08:02:58  completed   

   discount_pct  order_amount  delivery_minutes  
0            15        127.50              19.0  
1   

In [48]:
%run src/build_rfm.py

INSTAMART PART B - RFM ANALYSIS

FIRST 10 CUSTOMERS - RFM TABLE
  customer_id  recency  frequency  monetary
0       C0001       14          4    512.85
1       C0002       28          7   4644.90
2       C0003        7         22   3644.85
3       C0004        0          5    369.00
4       C0005      147          2    324.00
5       C0006       34          1     90.00
6       C0007        3         18   2600.00
7       C0008        3         15   3096.50
8       C0009       19         11   8424.80
9       C0010       66          4    450.00

RFM SUMMARY
           recency    frequency      monetary
count  1175.000000  1175.000000   1175.000000
mean     33.558298     9.455319   2552.058553
std      38.433705     8.009971   2367.305104
min       0.000000     1.000000     12.000000
25%       6.000000     3.000000    505.450000
50%      19.000000     5.000000   2286.500000
75%      46.000000    17.000000   3751.525000
max     180.000000    33.000000  14766.550000

RFM VALIDATION
Total cus

In [49]:
%run src/kmeans_analysis.py

INSTAMART PART B - K-MEANS ANALYSIS

RFM FEATURES STANDARDISED SUCCESSFULLY!


C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(
C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Window


K-MEANS COMPARISON
 K  Inertia  Silhouette Score
 2  1820.76             0.465
 3  1268.23             0.445
 4   743.07             0.526
 5   617.37             0.480
 6   503.17             0.482
 7   408.46             0.416
 8   366.50             0.412

K-MEANS ANALYSIS COMPLETED!
File: outputs/kmeans_metrics.csv


C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


In [50]:
%run src/segment_profiling.py

INSTAMART PART B - CUSTOMER SEGMENT PROFILING

CUSTOMER SEGMENT PROFILES
            customer_count  mean_recency  mean_frequency  mean_monetary
segment_id                                                             
0                      436         28.86            3.77        1057.62
1                      138         24.12            6.77        7129.70
2                      419          8.77           19.40        3410.02
3                      182        109.04            2.20         685.99

PCA EXPLAINED VARIANCE
Component 1: 0.634
Component 2: 0.218
Total explained variance: 0.852

SEGMENT PROFILING COMPLETED!
File: outputs/segment_profile.csv


C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


In [51]:
%run src/save_customer_segments.py

INSTAMART PART B - SAVING CUSTOMER SEGMENTS

FIRST 10 CUSTOMER SEGMENTS
customer_id  recency  frequency  monetary  segment_id                 segment_name  is_premium
      C0001       14          4    512.85           0  Regular Low-Spend Customers           0
      C0002       28          7   4644.90           1 Premium High-Value Customers           1
      C0003        7         22   3644.85           2     Loyal Frequent Customers           0
      C0004        0          5    369.00           0  Regular Low-Spend Customers           0
      C0005      147          2    324.00           3   At-Risk Inactive Customers           0
      C0006       34          1     90.00           0  Regular Low-Spend Customers           0
      C0007        3         18   2600.00           2     Loyal Frequent Customers           0
      C0008        3         15   3096.50           2     Loyal Frequent Customers           0
      C0009       19         11   8424.80           1 Premium High-Value 

C:\Users\divya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=5.
  warnings.warn(


In [52]:
%run src/build_lapse_dataset.py

INSTAMART PART B - LAPSE PREDICTION
STEP 1: BUILD LAPSE COHORT

LAPSE COHORT SUMMARY
Total eligible customers: 1156

LAPSED / NOT LAPSED COUNTS
lapsed
0    713
1    443
Name: count, dtype: int64

LAPSED CUSTOMER PERCENTAGE
38.32 %

FIRST 10 CUSTOMERS
customer_id  lapsed
      C0967       1
      C1117       0
      C0569       0
      C0776       0
      C1122       0
      C0674       0
      C0326       1
      C0084       0
      C1039       0
      C1152       0

LAPSE COHORT CREATED SUCCESSFULLY!
File: outputs/lapse_cohort.csv


In [54]:
%run src/build_customer_features.py

INSTAMART PART B - CUSTOMER FEATURES

FEATURE DATASET SHAPE
(1156, 12)

FIRST 10 CUSTOMERS
customer_id  recency  frequency  monetary  avg_discount  avg_delivery  cancellation_rate favourite_category      city  age gender  tenure_days
      C0001       33          2    378.00     10.000000     28.000000           0.000000            Grocery   Chennai 35.0   male          597
      C0002       22          6   3946.90      6.666667     21.916667           0.000000        Electronics     Delhi 31.0   male          562
      C0003        1         20   3264.05     12.750000     24.200000           0.000000            Grocery Bengaluru 18.0 female          195
      C0004       42          3    272.25     11.666667     35.666667           0.400000             Snacks Bengaluru 24.0 female          437
      C0005      117          2    324.00     25.000000     26.000000           0.333333              Dairy Bengaluru 25.0 female          335
      C0006        4          1     90.00     40.00

In [55]:
%run src/prepare_lapse_dataset.py

INSTAMART PART B - PREPARE LAPSE DATASET

FINAL DATASET SHAPE
(1156, 13)

FIRST 10 ROWS
customer_id  recency  frequency  monetary  avg_discount  avg_delivery  cancellation_rate favourite_category      city  age gender  tenure_days  lapsed
      C0001       33          2    378.00     10.000000     28.000000           0.000000            Grocery   Chennai 35.0   male          597       0
      C0002       22          6   3946.90      6.666667     21.916667           0.000000        Electronics     Delhi 31.0   male          562       0
      C0003        1         20   3264.05     12.750000     24.200000           0.000000            Grocery Bengaluru 18.0 female          195       0
      C0004       42          3    272.25     11.666667     35.666667           0.400000             Snacks Bengaluru 24.0 female          437       0
      C0005      117          2    324.00     25.000000     26.000000           0.333333              Dairy Bengaluru 25.0 female          335       1
      

In [56]:
%run src/split_lapse_data.py

INSTAMART PART B - TRAIN TEST SPLIT

TRAINING DATA SHAPE
(924, 11)

TESTING DATA SHAPE
(232, 11)

TRAINING TARGET DISTRIBUTION
lapsed
0    570
1    354
Name: count, dtype: int64

TESTING TARGET DISTRIBUTION
lapsed
0    143
1     89
Name: count, dtype: int64

TOTAL TRAINING ROWS: 924
TOTAL TESTING ROWS: 232

TRAIN TEST SPLIT COMPLETED!


In [57]:
%run src/preprocess_lapse_data.py

INSTAMART PART B - DATA PREPROCESSING

TRAINING PROCESSED SHAPE
(924, 19)

TESTING PROCESSED SHAPE
(232, 19)

MISSING VALUES IN ORIGINAL FEATURES
recency               0
frequency             0
monetary              0
avg_discount          0
avg_delivery          0
cancellation_rate     0
favourite_category    0
city                  0
age                   0
gender                0
tenure_days           0
dtype: int64

CHECK PROCESSED DATA
Training missing values: 0
Testing missing values: 0

FREQUENCY SCALER MEAN
8.114718614718615

FULL COHORT FREQUENCY MEAN
8.028546712802768

PREPROCESSING COMPLETED SUCCESSFULLY!


In [58]:
%run src/baseline_model.py

INSTAMART PART B - BASELINE MODEL

BASELINE ACCURACY
0.6164

LAPSED CUSTOMER F1 SCORE
0.0

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.62      1.00      0.76       143
           1       0.00      0.00      0.00        89

    accuracy                           0.62       232
   macro avg       0.31      0.50      0.38       232
weighted avg       0.38      0.62      0.47       232


BASELINE MODEL COMPLETED!


In [59]:
%run src/logistic_regression_model.py

INSTAMART PART B - LOGISTIC REGRESSION

BEST LOGISTIC REGRESSION PARAMETERS
{'model__C': 100}

BEST CROSS-VALIDATION F1 SCORE
0.6506

LOGISTIC REGRESSION TEST RESULTS
Accuracy: 0.7284
Precision: 0.6444
Recall: 0.6517
F1 Score: 0.648

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.78      0.78      0.78       143
           1       0.64      0.65      0.65        89

    accuracy                           0.73       232
   macro avg       0.71      0.71      0.71       232
weighted avg       0.73      0.73      0.73       232


LOGISTIC REGRESSION COMPLETED!


In [60]:
%run src/random_forest_model.py

INSTAMART PART B - RANDOM FOREST

BEST RANDOM FOREST PARAMETERS
{'model__max_depth': 8, 'model__min_samples_leaf': 10}

BEST CROSS-VALIDATION F1 SCORE
0.6811

RANDOM FOREST TEST RESULTS
Accuracy: 0.7371
Precision: 0.6591
Recall: 0.6517
F1 Score: 0.6554

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       143
           1       0.66      0.65      0.66        89

    accuracy                           0.74       232
   macro avg       0.72      0.72      0.72       232
weighted avg       0.74      0.74      0.74       232


RANDOM FOREST COMPLETED!


In [ ]:
%run src/oversampling_model.py

In [62]:
%run src/save_lapse_model.py

Balanced training counts:
lapsed
0    570
1    570
Name: count, dtype: int64

Model saved successfully!
Model reloaded successfully!

SUCCESS: Predictions match exactly!

FINAL MODEL SAVING COMPLETED!


In [63]:
%run src/premium_retention_list.py

PREMIUM CUSTOMER RETENTION LIST
customer_id  lapse_probability  actually_lapsed
      C0930           0.670191                0
      C0862           0.646898                0
      C1000           0.630894                0
      C1112           0.629366                0
      C0944           0.596449                0
      C0553           0.587494                0
      C0188           0.583205                1
      C0060           0.568153                0
      C0525           0.566409                1
      C0831           0.557253                0

Total Premium Customers in Test Set: 40
Top 10 Customers: 10
Actually Lapsed Customers in Top 10: 2

Retention list saved successfully!
File: outputs/premium_retention_list.csv


In [1]:
import pandas as pd

# Load support tickets
tickets = pd.read_csv("data/raw/tickets.csv")

print("SUPPORT TICKET DATASET")
print("=" * 50)

print("Total tickets:", len(tickets))

print("\nDataset shape:")
print(tickets.shape)

print("\nColumn names:")
print(tickets.columns.tolist())

print("\nFirst 5 tickets:")
print(tickets.head())

# Check empty tickets
empty_tickets = tickets["text"].isna() | (
    tickets["text"].astype(str).str.strip() == ""
)

print("\nEmpty tickets:", empty_tickets.sum())

# Check extremely long tickets
MAX_INPUT_TOKENS = 400

def est_tokens(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

tickets["estimated_tokens"] = tickets["text"].apply(est_tokens)

long_tickets = tickets["estimated_tokens"] > MAX_INPUT_TOKENS

print("Long tickets:", long_tickets.sum())

print("\nTickets exceeding token limit:")
print(
    tickets.loc[
        long_tickets,
        ["ticket_id", "estimated_tokens"]
    ]
)

SUPPORT TICKET DATASET
Total tickets: 164

Dataset shape:
(164, 5)

Column names:
['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']

First 5 tickets:
  ticket_id customer_id    order_id  \
0     T0024       C0486  ORD-105035   
1     T0073       C0080  ORD-100816   
2     T0095       C1087  ORD-111156   
3     T0159       C1172         NaN   
4     T0080       C0910  ORD-109516   

                                                text true_category  
0  There is an unauthorized transaction on my car...         fraud  
1  The invoice for ORD-100816 shows a wrong amoun...       billing  
2  The payment page shows an error and the app fr...     technical  
3                                                NaN          none  
4  I was charged for order ORD-109516 that I neve...         fraud  

Empty tickets: 4
Long tickets: 4

Tickets exceeding token limit:
    ticket_id  estimated_tokens
10      T0163               564
78      T0162               564
143     T0164          

## Part C – Privacy-Aware Support Ticket Triage

In [2]:
%run src/ticket_guardrails.py

INPUT GUARDRAIL RESULTS
status
eligible         156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64

Total tickets: 164

Empty tickets:
    ticket_id         status
3       T0159  skipped_empty
98      T0160  skipped_empty
128     T0157  skipped_empty
148     T0158  skipped_empty

Long tickets:
    ticket_id  estimated_tokens        status
10      T0163               564  flagged_long
78      T0162               564  flagged_long
143     T0164               564  flagged_long
158     T0161               564  flagged_long

Eligible tickets: 156

INPUT GUARDRAILS COMPLETED!


In [3]:
%run src/redact_tickets.py

PERSONAL INFORMATION REDACTION RESULTS
Email substitutions: 55
Phone substitutions: 91
Order ID substitutions: 132
Total substitutions: 278

PRIVACY VALIDATION
Remaining emails: 0
Remaining phone numbers: 0
Remaining order IDs: 0

SUCCESS: No sensitive information remains!

Redacted tickets saved successfully!
File: outputs/redacted_tickets.csv


In [4]:
%run src/triage_prompts.py

ZERO-SHOT PROMPT VALIDATED!
FEW-SHOT PROMPT VALIDATED!
All four categories are present!
JSON response format validated!

PROMPT CREATION COMPLETED!


In [5]:
%run src/triage_retry.py

RETRY FUNCTION CREATED SUCCESSFULLY!


In [6]:
import mock_llm
import inspect

print("CHAT FUNCTION:")
print(inspect.signature(mock_llm.chat))

print("\nRESET FUNCTION:")
print(inspect.signature(mock_llm.reset))

print("\nAVAILABLE ERROR CLASSES:")
print([
    name for name in dir(mock_llm)
    if "Error" in name or "Transient" in name
])

print("\nCHAT FUNCTION SOURCE:")
print(inspect.getsource(mock_llm.chat))

CHAT FUNCTION:
(messages, tools=None)

RESET FUNCTION:
()

AVAILABLE ERROR CLASSES:
['TransientLLMError']

CHAT FUNCTION SOURCE:
def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")



In [7]:
%run src/triage_validation.py

RESPONSE VALIDATION FUNCTION CREATED SUCCESSFULLY!


In [8]:
from src.triage_validation import validate_response

test_responses = [
    '{"category": "billing", "reason": "Payment issue"}',
    'This ticket belongs to delivery category.',
    'I cannot classify this ticket.',
    '{"category": "unknown", "reason": "Other issue"}'
]

for response in test_responses:
    result = validate_response(response)
    print("Input:", response)
    print("Output:", result)
    print("-" * 60)

RESPONSE VALIDATION FUNCTION CREATED SUCCESSFULLY!
Input: {"category": "billing", "reason": "Payment issue"}
Output: {'status': 'classified_json', 'category': 'billing', 'reason': 'Payment issue'}
------------------------------------------------------------
Input: This ticket belongs to delivery category.
Output: {'status': 'classified_keyword', 'category': 'delivery', 'reason': ''}
------------------------------------------------------------
Input: I cannot classify this ticket.
Output: {'status': 'flagged_invalid_output', 'category': '', 'reason': 'Invalid model response. Human review required.'}
------------------------------------------------------------
Input: {"category": "unknown", "reason": "Other issue"}
Output: {'status': 'flagged_invalid_output', 'category': '', 'reason': 'Invalid model response. Human review required.'}
------------------------------------------------------------


In [9]:
%run src/triage_validation.py

RESPONSE VALIDATION FUNCTION CREATED SUCCESSFULLY!


In [10]:
from src.triage_validation import validate_response

test_responses = [
    '{"category": "billing", "reason": "Payment issue"}',
    'This ticket belongs to delivery category.',
    'I cannot classify this ticket.',
    '{"category": "unknown", "reason": "Other issue"}'
]

for response in test_responses:
    print(validate_response(response))

{'status': 'classified_json', 'category': 'billing', 'reason': 'Payment issue'}
{'status': 'classified_keyword', 'category': 'delivery', 'reason': ''}
{'status': 'flagged_invalid_output', 'category': '', 'reason': 'Invalid model response. Human review required.'}
{'status': 'flagged_invalid_output', 'category': '', 'reason': 'Invalid model response. Human review required.'}


In [11]:
import mock_llm
import inspect

print("CLASSIFICATION FUNCTION:")
print(inspect.getsource(mock_llm._classify))

CLASSIFICATION FUNCTION:
def _classify(messages):
    system, user = messages[0]["content"], messages[-1]["content"]
    _state["calls"] += 1
    if _state["calls"] % 9 == 0:
        raise TransientLLMError("503 server busy")
    _state["ok"] += 1
    order = ["fraud", "technical", "billing", "delivery"] if "EXAMPLES:" in system else ["billing", "fraud", "technical", "delivery"]
    w = _words(user)
    cat = next((c for c in order if KW[c] & w), "technical")
    if _state["ok"] % 13 == 0:
        return _resp(f"The customer appears to have a {cat}-related problem.", messages=messages)
    if _state["ok"] % 17 == 0:
        return _resp(json.dumps({"category": "shipping", "reason": "parcel movement"}), messages=messages)
    return _resp(json.dumps({"category": cat, "reason": "keyword match"}), messages=messages)



In [12]:
import inspect
import mock_llm

print("RESPONSE FUNCTION:")
print(inspect.getsource(mock_llm._resp))

RESPONSE FUNCTION:
def _resp(content=None, tool_calls=None, finish="stop", messages=()):
    prompt = sum(est_tokens(m.get("content") or json.dumps(m.get("tool_calls", ""), default=str)) for m in messages)
    comp = est_tokens(content or json.dumps(tool_calls or "", default=str))
    msg = NS(role="assistant", content=content, tool_calls=[NS(**tc) for tc in (tool_calls or [])])
    return NS(choices=[NS(message=msg, finish_reason=finish)],
              usage=NS(prompt_tokens=prompt, completion_tokens=comp, total_tokens=prompt + comp))



In [ ]:
import pandas as pd
import src.triage_prompts as prompts

# Inspect redacted ticket data
df = pd.read_csv("outputs/redacted_tickets.csv")

print("REDACTED CSV COLUMNS:")
print(df.columns.tolist())

print("\nFIRST TWO ROWS:")
print(df.head(2).to_string())

print("\nPROMPT VARIABLES:")
print([
    name for name in dir(prompts)
    if name.isupper() and "PROMPT" in name
])

print("\nZERO-SHOT PROMPT:")
print(prompts.ZERO_SHOT_PROMPT)

print("\nFEW-SHOT PROMPT:")
print(prompts.FEW_SHOT_PROMPT)

In [14]:
import mock_llm
import inspect

print(inspect.getsource(mock_llm.est_tokens))

def est_tokens(text):
    return math.ceil(len((text or "").split()) * 1.3)



In [19]:
%run src/compare_prompts.py

Ticket T0002 | Retry number: 0 | Wait: 0.0460 seconds
Ticket T0116 | Retry number: 0 | Wait: 0.0458 seconds
Ticket T0153 | Retry number: 0 | Wait: 0.0466 seconds
Ticket T0039 | Retry number: 0 | Wait: 0.0553 seconds
Ticket T0152 | Retry number: 0 | Wait: 0.0456 seconds
Ticket T0043 | Retry number: 0 | Wait: 0.0455 seconds
Ticket T0111 | Retry number: 0 | Wait: 0.0493 seconds
Ticket T0155 | Retry number: 0 | Wait: 0.0544 seconds
Ticket T0100 | Retry number: 0 | Wait: 0.0488 seconds
Ticket T0068 | Retry number: 0 | Wait: 0.0451 seconds
Ticket T0135 | Retry number: 0 | Wait: 0.0522 seconds
Ticket T0009 | Retry number: 0 | Wait: 0.0531 seconds
Ticket T0067 | Retry number: 0 | Wait: 0.0519 seconds
Ticket T0106 | Retry number: 0 | Wait: 0.0521 seconds
Ticket T0097 | Retry number: 0 | Wait: 0.0529 seconds
Ticket T0003 | Retry number: 0 | Wait: 0.0549 seconds
Ticket T0019 | Retry number: 0 | Wait: 0.0510 seconds
Ticket T0086 | Retry number: 0 | Wait: 0.0554 seconds
Ticket T0065 | Retry number:

In [20]:
import pandas as pd

segments = pd.read_csv("outputs/customer_segments.csv")

print("COLUMNS:")
print(segments.columns.tolist())

print("\nFIRST FIVE ROWS:")
print(segments.head().to_string())

print("\nPREMIUM COLUMN COUNTS:")
print(segments["is_premium"].value_counts(dropna=False))

COLUMNS:
['customer_id', 'recency', 'frequency', 'monetary', 'segment_id', 'segment_name', 'is_premium']

FIRST FIVE ROWS:
  customer_id  recency  frequency  monetary  segment_id                  segment_name  is_premium
0       C0001       14          4    512.85           0   Regular Low-Spend Customers           0
1       C0002       28          7   4644.90           1  Premium High-Value Customers           1
2       C0003        7         22   3644.85           2      Loyal Frequent Customers           0
3       C0004        0          5    369.00           0   Regular Low-Spend Customers           0
4       C0005      147          2    324.00           3    At-Risk Inactive Customers           0

PREMIUM COLUMN COUNTS:
is_premium
0    1037
1     138
Name: count, dtype: int64


In [21]:
%run src/triage_final.py


ZERO-SHOT BUDGET DEMONSTRATION
Ticket T0002 | Retry number: 0 | Wait: 0.0543 seconds
Ticket T0116 | Retry number: 0 | Wait: 0.0552 seconds
Ticket T0153 | Retry number: 0 | Wait: 0.0528 seconds
Ticket T0039 | Retry number: 0 | Wait: 0.0480 seconds
Ticket T0152 | Retry number: 0 | Wait: 0.0562 seconds
Ticket T0043 | Retry number: 0 | Wait: 0.0543 seconds
Ticket T0111 | Retry number: 0 | Wait: 0.0555 seconds

Budget demonstration saved!
File: outputs/triage_budget_demo.csv
Token budget: 6000
Tokens used: 6047

Budget run status counts:
status
budget_exhausted          99
classified_json           50
skipped_empty              4
flagged_long               4
classified_keyword         4
flagged_invalid_output     3
Name: count, dtype: int64

Budget exhausted tickets: 99


FINAL FEW-SHOT TICKET CLASSIFICATION
Ticket T0002 | Retry number: 0 | Wait: 0.0568 seconds
Ticket T0116 | Retry number: 0 | Wait: 0.0481 seconds
Ticket T0153 | Retry number: 0 | Wait: 0.0504 seconds
Ticket T0039 | Retry n

In [22]:
import pandas as pd

df = pd.read_csv("outputs/triage_results.csv")

human_statuses = [
    "skipped_empty",
    "flagged_long",
    "flagged_invalid_output",
    "budget_exhausted"
]

# Check required columns
expected_columns = [
    "ticket_id", "customer_id", "status", "category",
    "segment_name", "priority", "route", "tokens_used"
]

assert df.columns.tolist() == expected_columns
assert len(df) == 164

# Check human review routing
human = df[df["status"].isin(human_statuses)]
assert (human["route"] == "human_review").all()
assert (human["priority"] == "NORMAL").all()

# Check fraud routing
fraud = df[
    (df["status"].isin(["classified_json", "classified_keyword"])) &
    (df["category"] == "fraud")
]
assert (fraud["route"] == "security_team").all()
assert (fraud["priority"] == "HIGH").all()

# Check premium routing
premium = df[
    (df["status"].isin(["classified_json", "classified_keyword"])) &
    (df["category"] != "fraud") &
    (df["segment_name"] == "Premium High-Value Customers")
]
assert (premium["route"] == "premium_desk").all()
assert (premium["priority"] == "HIGH").all()

print("ALL FINAL ROUTING CHECKS PASSED!")
print("Total records:", len(df))
print("Required columns: PASS")
print("Human review routing: PASS")
print("Fraud routing: PASS")
print("Premium routing: PASS")

ALL FINAL ROUTING CHECKS PASSED!
Total records: 164
Required columns: PASS
Human review routing: PASS
Fraud routing: PASS
Premium routing: PASS


## Part D – Policy RAG and ReAct Refund Agent

In [23]:
%run src/policy_chunking.py

delivery_policy.txt: 3 chunks
escalation_policy.txt: 3 chunks
payment_policy.txt: 3 chunks
privacy_policy.txt: 2 chunks
refund_policy.txt: 4 chunks

TOTAL CHUNKS: 15
Saved: outputs\policy_chunks.json

FIRST CHUNK:
{'source': 'delivery_policy.txt', 'start_word': 0, 'end_word': 39, 'text': 'Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,'}

TASK 1 COMPLETED!


In [24]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [25]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)
print("Installation successful!")

Scikit-learn version: 1.7.2
Installation successful!


In [26]:
%run src/policy_retrieval.py

QUERY: What is the refund policy for cancelled orders?

TOP 3 RETRIEVED POLICY CHUNKS:

Result 1
Source: refund_policy.txt
Similarity: 0.5728
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer
------------------------------------------------------------
Result 2
Source: refund_policy.txt
Similarity: 0.1554
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are
------------------------------------------------------------
Result 3
Source: delivery_policy.txt
Similarity: 0.1381
Text: of INR 25 is charged on orders below INR 199, and delivery is free on orders of INR 199 or more. During heavy rain the promised time may extend to 60 minut

In [27]:
results = retrieve("What happens when my delivery is delayed?", k=3)

for result in results:
    print("Source:", result["source"])
    print("Similarity:", result["similarity"])
    print("Text:", result["text"])
    print("-" * 50)

Source: delivery_policy.txt
Similarity: 0.2998
Text: Delivery Policy. Standard delivery is promised within 30 minutes of order confirmation in all supported cities. The supported cities are Chennai, Mumbai, Bengaluru, Delhi, Hyderabad and Pune. A delivery fee of INR 25 is charged on orders below INR 199,
--------------------------------------------------
Source: payment_policy.txt
Similarity: 0.2199
Text: Payment Policy. Customers can pay using UPI, debit card, credit card or cash on delivery. Cash on delivery is available only for orders up to INR 2500. If a payment fails but money is deducted, the amount is returned automatically
--------------------------------------------------
Source: refund_policy.txt
Similarity: 0.1464
Text: money is returned. Approved refunds are credited to the original payment method within 5 working days. A refund request must be raised within 7 days of the delivery date.
--------------------------------------------------


In [29]:
%run src/policy_assistant.py

QUESTION:
What is the refund policy for cancelled orders?

ANSWER:
Orders with status cancelled were never charged, so no refund applies to them.

SOURCES:
- refund_policy.txt
- payment_policy.txt


In [30]:
import pandas as pd

eval_df = pd.read_csv("data/raw/policy_eval.csv")

print("Dataset shape:", eval_df.shape)
print("\nColumns:")
print(eval_df.columns.tolist())

print("\nAll evaluation questions:")
display(eval_df)

Dataset shape: (14, 3)

Columns:
['question', 'expected_doc', 'expected_fact']

All evaluation questions:


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days


In [34]:
%run src/evaluate_policy.py


40 WORDS WITH 10-WORD OVERLAP
Total chunks: 15

Evaluation Results:
Answerable questions: 12
Expected document retrieved: 12 / 12
Expected fact found: 12 / 12
Unanswerable fallback: 2 / 2

40 WORDS WITH ZERO OVERLAP
Total chunks: 13

Evaluation Results:
Answerable questions: 12
Expected document retrieved: 12 / 12
Expected fact found: 10 / 12
Unanswerable fallback: 2 / 2

FINAL COMPARISON:
                     Chunking  Total chunks Document retrieval Expected fact Fallback
40 WORDS WITH 10-WORD OVERLAP            15              12/12         12/12      2/2
   40 WORDS WITH ZERO OVERLAP            13              12/12         10/12      2/2

Saved: outputs/policy_evaluation.csv

TASK 4 EVALUATION COMPLETED!


In [35]:
%run src/refund_tools.py

Number of tools: 5
get_order
search_policy
calculate_refund
issue_refund
escalate_to_human


In [36]:
%run src/refund_tools.py

Number of tools: 5
get_order
search_policy
calculate_refund
issue_refund
escalate_to_human
TOOL_MAP created successfully!
Available tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human']


In [37]:
from refund_tools import get_order

result = get_order("ORD-1001")

print(result)

Number of tools: 5
get_order
search_policy
calculate_refund
issue_refund
escalate_to_human
TOOL_MAP created successfully!
Available tools: ['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human']
{'error': 'order not found'}


In [38]:
import sqlite3

connection = sqlite3.connect("data/instamart.db")

cursor = connection.cursor()

cursor.execute("""
    SELECT order_id, status, order_amount, delivery_minutes
    FROM orders
    LIMIT 5
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

connection.close()

('ORD-110093', 'completed', 127.5, 19.0)
('ORD-111396', 'completed', 1598.0, 25.0)
('ORD-102006', 'cancelled', 21.75, None)
('ORD-105921', 'completed', 86.4, 33.0)
('ORD-108115', 'completed', 104.4, 21.0)


In [39]:
result = get_order("ORD-110093")

print(result)

{'order_id': 'ORD-110093', 'status': 'completed', 'order_amount': 127.5, 'delivery_minutes': 19.0}


In [40]:
from refund_tools import search_policy

result = search_policy("refund for late delivery")

print(result)

{'results': [{'source': 'refund_policy.txt', 'score': 0.3866, 'text': 'receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR'}, {'source': 'refund_policy.txt', 'score': 0.2531, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'payment_policy.txt', 'score': 0.1235, 'text': 'Payment Policy. Customers can pay using UPI, debit card, credit card or cash on delivery. Cash on delivery is available only for orders up to INR 2500. If a payment fails but money is deducted, the amount is returned automatically'}]}


In [41]:
from refund_tools import calculate_refund

# Test A: Late delivery
print("Test A:", calculate_refund(127.50, 60))

# Test B: On-time delivery
print("Test B:", calculate_refund(127.50, 30))

Test A: {'refund_inr': 127.5}
Test B: {'refund_inr': 0.0}


In [42]:
from refund_tools import issue_refund

result = issue_refund("ORD-110093", 127.50)

print(result)

{'refund_inr': 127.5}


In [43]:
with open("outputs/refund_ledger.jsonl", "r", encoding="utf-8") as file:
    print(file.read())

{"timestamp": "2026-11-01T22:38:31.075022", "tool_name": "issue_refund", "order_id": "ORD-110093", "refund_inr": 127.5}



In [44]:
from refund_tools import escalate_to_human

result = escalate_to_human("ORD-111396", 1598.00)

print(result)

{'refund_inr': 1598.0}


In [45]:
with open("outputs/refund_ledger.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        print(line.strip())

{"timestamp": "2026-11-01T22:38:31.075022", "tool_name": "issue_refund", "order_id": "ORD-110093", "refund_inr": 127.5}
{"timestamp": "2026-11-01T22:39:46.112899", "tool_name": "escalate_to_human", "order_id": "ORD-111396", "refund_inr": 1598.0}


In [46]:
import mock_llm
import inspect

print("CHAT FUNCTION:")
print(inspect.signature(mock_llm.chat))

print("\nCHAT FUNCTION SOURCE:")
print(inspect.getsource(mock_llm.chat))

CHAT FUNCTION:
(messages, tools=None)

CHAT FUNCTION SOURCE:
def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")



### Part D – Agent Scenario Generation and Recovery Tests

In [ ]:
from pathlib import Path

agent_source = Path("src/agent.py")
if not agent_source.is_file():
    raise FileNotFoundError("src/agent.py is missing. Restore the project source file before continuing.")

print("Using the maintained scenario-builder source:", agent_source)
print("The notebook will not overwrite src/agent.py.")


In [ ]:
import sys
if "src" not in sys.path:
    sys.path.insert(0, "src")

from agent import build_scenarios

scenarios = build_scenarios()

print("Scenario file created successfully!")
display(scenarios)


In [57]:
import inspect
import mock_llm

print("CHAT FUNCTION SIGNATURE:")
print(inspect.signature(mock_llm.chat))

print("\nMOCK LLM CHAT IMPLEMENTATION:")
print(inspect.getsource(mock_llm.chat))

CHAT FUNCTION SIGNATURE:
(messages, tools=None)

MOCK LLM CHAT IMPLEMENTATION:
def chat(messages, tools=None):
    head = messages[0]["content"].upper()
    if "CLASSIFY" in head:
        return _classify(messages)
    if "POLICY ASSISTANT" in head:
        return _policy(messages)
    if "AGENT" in head:
        return _agent(messages, tools)
    raise ValueError("first message must name one of: CLASSIFY, POLICY ASSISTANT, AGENT")



In [58]:
import inspect
import mock_llm

print(inspect.getsource(mock_llm._agent))

def _agent(messages, tools):
    user = next(m["content"] for m in messages if m["role"] == "user")
    tool_msgs = [m for m in messages if m["role"] == "tool"]
    call = lambda n, a: _resp(None, [{"id": f"call_{len(tool_msgs)+1}", "name": n, "arguments": a}], "tool_calls", messages)
    if "[LOOP]" in user:
        return call("search_policy", {"query": "refund policy"})
    if not tool_msgs:
        m = re.search(r"ORD-\d{6}", user)
        return call("get_order", {"order_id": m.group(0)}) if m else _resp("NEEDS_INFO: please share the order id.", messages=messages)
    last = tool_msgs[-1]
    data = json.loads(last["content"])
    name = last["name"]
    if name == "get_order":
        if "error" in data:
            return _resp("NOT_FOUND: no such order.", messages=messages)
        return call("search_policy", {"query": "refund policy late delivery"})
    order = json.loads(next(m for m in tool_msgs if m["name"] == "get_order")["content"])
    if name == "search_policy":
      

In [59]:
import inspect
import mock_llm

print(inspect.getsource(mock_llm._resp))

def _resp(content=None, tool_calls=None, finish="stop", messages=()):
    prompt = sum(est_tokens(m.get("content") or json.dumps(m.get("tool_calls", ""), default=str)) for m in messages)
    comp = est_tokens(content or json.dumps(tool_calls or "", default=str))
    msg = NS(role="assistant", content=content, tool_calls=[NS(**tc) for tc in (tool_calls or [])])
    return NS(choices=[NS(message=msg, finish_reason=finish)],
              usage=NS(prompt_tokens=prompt, completion_tokens=comp, total_tokens=prompt + comp))



In [60]:
import importlib
import refund_agent

importlib.reload(refund_agent)

print("Refund agent loaded successfully!")

Refund agent loaded successfully!


In [61]:
results = refund_agent.run_all_scenarios()

S1: REFUND | Calls: 5 | Tokens: 1452
S2: REFUND | Calls: 5 | Tokens: 1452
S3: ESCALATE | Calls: 5 | Tokens: 1439
S4: DENIED | Calls: 4 | Tokens: 1049
S5: DENIED | Calls: 3 | Tokens: 670
S6: NOT_FOUND | Calls: 2 | Tokens: 294
S7: FALLBACK_MAX_ITERATIONS | Calls: 6 | Tokens: 3651


In [62]:
import inspect
print(inspect.getsource(refund_agent.run_agent))

def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    messages = [
        {
            "role": "system",
            "content": AGENT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    steps = []
    llm_calls = 0
    tokens_used = 0
    final_answer = None
    status = None

    while llm_calls < max_iterations:

        response = mock_llm.chat(
            messages,
            tools=TOOLS_SPEC
        )

        llm_calls += 1

        tokens_used += response.usage.total_tokens

        assistant_message = response.choices[0].message

        tool_calls = assistant_message.tool_calls

        # Check whether the model returned a final answer
        if not tool_calls:

            final_answer = assistant_message.content or ""

            if ":" in final_answer:
                status = final_answer.split(":")[0].strip()
            else:

In [63]:
import importlib
import refund_agent

importlib.reload(refund_agent)

scenarios = pd.read_csv("data/raw/agent_scenarios.csv")

s1 = scenarios[scenarios["scenario_id"] == "S1"].iloc[0]

token_test = refund_agent.run_agent(
    user_message=s1["message"],
    scenario_id="S1_TOKEN_TEST",
    token_limit=200
)

print("Status:", token_test["status"])
print("Final answer:", token_test["final_answer"])
print("LLM calls:", token_test["llm_calls"])
print("Tokens:", token_test["tokens_used"])
print("Executed tools:", token_test["steps"])

Status: FALLBACK_TOKEN_LIMIT
Final answer: FALLBACK_TOKEN_LIMIT: Agent stopped safely. Scenario: S1_TOKEN_TEST. Tools executed: 1. Please review the request and order details manually.
LLM calls: 2
Tokens: 324
Executed tools: [{'tool': 'get_order', 'args': {'order_id': 'ORD-100048'}, 'observation': {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65.0}}]


In [64]:
import pandas as pd

scenarios = pd.read_csv("data/raw/agent_scenarios.csv")

s7 = scenarios[scenarios["scenario_id"] == "S7"].iloc[0]

result_s7 = refund_agent.run_agent(
    user_message=s7["message"],
    scenario_id="S7",
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
)

print("Status:", result_s7["status"])
print("Final answer:", result_s7["final_answer"])
print("LLM calls:", result_s7["llm_calls"])
print("Tokens:", result_s7["tokens_used"])
print("Tools executed:", len(result_s7["steps"]))

Status: FALLBACK_MAX_ITERATIONS
Final answer: FALLBACK_MAX_ITERATIONS: Agent stopped safely. Scenario: S7. Tools executed: 6. Please review the request and order details manually.
LLM calls: 6
Tokens: 3651
Tools executed: 6


In [65]:
timeout_result = refund_agent.run_agent(
    user_message="Refund for order ORD-100048 please.",
    scenario_id="TIMEOUT_TEST",
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=0
)

print("Status:", timeout_result["status"])
print("Final answer:", timeout_result["final_answer"])
print("LLM calls:", timeout_result["llm_calls"])
print("Tokens:", timeout_result["tokens_used"])
print("Tools executed:", len(timeout_result["steps"]))

Status: FALLBACK_TIMEOUT
Final answer: FALLBACK_TIMEOUT: Agent stopped safely. Scenario: TIMEOUT_TEST. Tools executed: 0. Please review the request and order details manually.
LLM calls: 0
Tokens: 0
Tools executed: 0


In [66]:
import inspect
import refund_agent

print(inspect.getsource(refund_agent.run_agent))

def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    import time

    start_time = time.monotonic()

    messages = [
        {
            "role": "system",
            "content": AGENT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    steps = []
    llm_calls = 0
    tokens_used = 0
    final_answer = None
    status = None

    def human_handoff(reason):
        return (
            f"{reason}: Agent stopped safely. "
            f"Scenario: {scenario_id}. "
            f"Tools executed: {len(steps)}. "
            "Please review the request and order details manually."
        )

    while True:

        # 1. TIME LIMIT CHECK
        if time.monotonic() - start_time >= time_limit_seconds:

            status = "FALLBACK_TIMEOUT"
            final_answer = human_handoff(
                "FALLBACK_TIMEOUT"
            )
            break

  

In [67]:
import importlib
import refund_agent

importlib.reload(refund_agent)

print("Refund agent loaded successfully!")
print("Resume function:", hasattr(refund_agent, "resume_agent"))

Checkpoint folder ready!
Refund agent loaded successfully!
Resume function: True


In [68]:
import refund_agent

print("Checkpoint function:", hasattr(refund_agent, "save_checkpoint"))
print("Resume function:", hasattr(refund_agent, "resume_agent"))
print("State folder:", refund_agent.STATE_DIR)

Checkpoint function: True
Resume function: True
State folder: state


In [69]:
import os

print("Checkpoint files:")

for file in os.listdir("state"):
    print(file)

Checkpoint files:


In [70]:
import importlib
import refund_agent

importlib.reload(refund_agent)

print("Agent loaded:", hasattr(refund_agent, "run_agent"))
print("Resume loaded:", hasattr(refund_agent, "resume_agent"))
print("Scenario builder:", hasattr(refund_agent, "build_scenarios"))

Checkpoint folder ready!
Refund agent module loaded successfully!
Agent loaded: True
Resume loaded: True
Scenario builder: True


In [71]:
scenarios = refund_agent.build_scenarios()

print("\nScenario count:", len(scenarios))
print("\nScenario IDs:")
print(scenarios["scenario_id"].tolist())

print("\nExpected outcomes:")
print(scenarios[["scenario_id", "expected_outcome"]])

Agent scenarios created successfully!
  scenario_id                                       user_message  \
0          S1  Please check refund eligibility for order ORD-...   
1          S2  Please check refund eligibility for order ORD-...   
2          S3  Please check refund eligibility for order ORD-...   
3          S4  Please check refund eligibility for order ORD-...   
4          S5  Please check refund eligibility for order ORD-...   
5          S6  Please check refund eligibility for order ORD-...   
6          S7                    [LOOP] Refund order ORD-100048.   

          expected_outcome    order_id  
0                   REFUND  ORD-100048  
1                   REFUND  ORD-100067  
2                 ESCALATE  ORD-100145  
3                   DENIED  ORD-100001  
4                   DENIED  ORD-100020  
5                NOT_FOUND  ORD-999999  
6  FALLBACK_MAX_ITERATIONS  ORD-100048  

Scenario count: 7

Scenario IDs:
['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7']

Expected ou

In [72]:
s1 = scenarios[scenarios["scenario_id"] == "S1"].iloc[0]

print("Scenario:", s1["scenario_id"])
print("Order ID:", s1["order_id"])
print("Expected:", s1["expected_outcome"])

Scenario: S1
Order ID: ORD-100048
Expected: REFUND


In [73]:
run_id = "S1_CRASH_RESUME_01"

checkpoint_path = f"state/{run_id}.json"

if os.path.exists(checkpoint_path):
    print("Checkpoint already exists. Stop and tell me.")
else:
    try:
        result = refund_agent.run_agent(
            user_message=s1["user_message"],
            scenario_id="S1",
            run_id=run_id,
            crash_after_tools=2
        )

    except RuntimeError as error:
        print("Crash simulation:", error)

    print("\nCheckpoint created:",
          os.path.exists(checkpoint_path))

Crash simulation: SIMULATED_CRASH: S1_CRASH_RESUME_01

Checkpoint created: True


In [74]:
import json

with open(
    "state/S1_CRASH_RESUME_01.json",
    "r",
    encoding="utf-8"
) as file:
    checkpoint = json.load(file)

print("Run ID:", checkpoint["run_id"])
print("Scenario:", checkpoint["scenario_id"])
print("Step number:", checkpoint["step_number"])
print("Tools executed:", checkpoint["tools_executed"])
print("LLM calls:", checkpoint["llm_calls"])
print("Tokens used:", checkpoint["tokens_used"])

Run ID: S1_CRASH_RESUME_01
Scenario: S1
Step number: 2
Tools executed: ['get_order', 'search_policy']
LLM calls: 2
Tokens used: 402


In [75]:
resumed_result = refund_agent.resume_agent(
    run_id="S1_CRASH_RESUME_01"
)

print("\n========== RESUMED AGENT RESULT ==========")
print("Status:", resumed_result["status"])
print("Final answer:", resumed_result["final_answer"])
print("Total LLM calls:", resumed_result["llm_calls"])
print("Total tokens:", resumed_result["tokens_used"])

print("\nTools executed:")
for step in resumed_result["steps"]:
    print(step["tool"], step["args"])
    

Checkpoint loaded successfully!
Run ID: S1_CRASH_RESUME_01
Scenario: S1
Previously executed tools: ['get_order', 'search_policy']

========== RESUMED AGENT RESULT ==========
Status: REFUND
Final answer: REFUND: INR 128.25 issued.
Total LLM calls: 5
Total tokens: 1647

Tools executed:
get_order {'order_id': 'ORD-100048'}
search_policy {'query': 'refund policy late delivery'}
calculate_refund {'order_amount': 128.25, 'delivery_minutes': 65.0}
issue_refund {'order_id': 'ORD-100048', 'refund_inr': 128.25}


In [76]:
import os

print("Refund ledger exists:",
      os.path.exists("outputs/refund_ledger.jsonl"))

Refund ledger exists: True


In [77]:
import json

ledger_path = "outputs/refund_ledger.jsonl"

with open(ledger_path, "r", encoding="utf-8") as file:
    ledger_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

s1_records = [
    record for record in ledger_records
    if record.get("order_id") == "ORD-100048"
]

print("S1 refund records:", len(s1_records))

for record in s1_records:
    print(record)

S1 refund records: 2
{'timestamp': '2026-11-01T22:58:37.192831', 'tool_name': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-01T23:43:53.397592', 'tool_name': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}


In [78]:
results_task9 = {}

for scenario_id in ["S5", "S6"]:

    scenario = scenarios[
        scenarios["scenario_id"] == scenario_id
    ].iloc[0]

    result = refund_agent.run_agent(
        user_message=scenario["user_message"],
        scenario_id=scenario_id
    )

    results_task9[scenario_id] = result

    print("\nScenario:", scenario_id)
    print("Status:", result["status"])
    print("LLM calls:", result["llm_calls"])
    print("Tokens:", result["tokens_used"])
    print("Tools:", [
        step["tool"] for step in result["steps"]
    ])


Scenario: S5
Status: DENIED
LLM calls: 3
Tokens: 802
Tools: ['get_order', 'search_policy']

Scenario: S6
Status: NOT_FOUND
LLM calls: 2
Tokens: 390
Tools: ['get_order']
